# CO7093 — Group 12 Coursework Solution
**Dataset:** UCI Diabetes 130-US hospitals (`diabetic_data.csv`)

**Group 12 members (anonymous)**
| University ID |
|---------------|
| sa1213 |
| nny3 |
| gm420 |

**Notebook:** `group12_solution.ipynb` (per brief: `group{N}_solution.ipynb`)

**How to run (local)**
1. Place `diabetic_data.csv` in the same folder as this notebook (project root).
2. Install: `pip install -r requirements.txt` (includes `pyspark`, `imbalanced-learn`).
3. **PySpark** needs a **Java 11+** JDK on `PATH` / `JAVA_HOME` (common on lab machines).
4. Run all cells top to bottom.

**How to run (Google Colab)**  
Use **Runtime → Run all**. Run the **“Colab setup”** cell first: it installs packages, OpenJDK 11, downloads the UCI zip, and copies `diabetic_data.csv` into `/content`. If the download fails, upload `diabetic_data.csv` manually when prompted.

**Colab RAM:** Part 3 defaults to **18,000 stratified rows** on Colab (**float32** dummies + memory cleanup). Part 2 auto-switches to **3-fold CV**, **80 trees** RF, shorter LR iterations. Full data: High-RAM runtime + `import os; os.environ["PART3_MAX_ROWS"]="0"` before Part 3.

**Parts**
- **Part 1:** Cleaning, missing data, EDA plots (Pandas / Matplotlib / Seaborn).
- **Part 2:** Three sklearn classifiers, stratified CV, **random undersampling** on training data, metrics.
- **Part 3:** PySpark pipeline on full data, **K-Means**, elbow + silhouette, **local classifiers** vs global model.


### Google Colab — run the next cell once per session
On your laptop / Jupyter locally, that cell **does nothing** (it only detects Colab).

In [ ]:
# Colab setup: installs deps, Java for Spark, and fetches diabetic_data.csv into /content
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    import shutil
    import subprocess
    import sys
    import urllib.request
    import zipfile
    from pathlib import Path

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "imbalanced-learn>=0.12",
            "pyspark>=3.5",
            "pyarrow>=14",
        ]
    )
    subprocess.run(
        ["apt-get", "-qq", "-y", "install", "openjdk-11-jdk-headless"],
        check=False,
    )
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ.get("PATH", "")

    data_zip = (
        "https://archive.ics.uci.edu/static/public/296/"
        "diabetes+130-us+hospitals+for+years+1999-2008.zip"
    )
    zpath = Path("/content/diabetes_130.zip")
    dest_csv = Path("/content/diabetic_data.csv")
    try:
        urllib.request.urlretrieve(data_zip, zpath)
        with zipfile.ZipFile(zpath, "r") as zf:
            zf.extractall("/content")
        hits = list(Path("/content").rglob("diabetic_data.csv"))
        if hits:
            shutil.copy(hits[0], dest_csv)
            with open(dest_csv, encoding="utf-8", errors="replace") as f:
                nlines = sum(1 for _ in f) - 1
            print("Colab: saved", dest_csv.resolve(), f"({nlines:,} data rows)")
        else:
            print("Zip extracted but diabetic_data.csv not found; upload manually (next).")
    except Exception as exc:
        print("Auto-download failed:", exc)
        print("Upload diabetic_data.csv using the file picker:")
        from google.colab import files

        uploaded = files.upload()
        if "diabetic_data.csv" in uploaded:
            Path("/content/diabetic_data.csv").write_bytes(uploaded["diabetic_data.csv"])
            print("Saved /content/diabetic_data.csv")
    print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
else:
    print("Not on Google Colab — skipped (use local diabetic_data.csv + venv).")


## Part 1 — Data exploration and preprocessing

In [ ]:
# Configuration and imports
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
rng = np.random.RandomState(RANDOM_STATE)

ROOT = Path.cwd()
DATA_PATH = ROOT / "diabetic_data.csv"
if not DATA_PATH.exists():
    DATA_PATH = Path("diabetic_data.csv")

print("Data path:", DATA_PATH.resolve())


In [ ]:
def load_raw(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    # Standardise placeholders used in this release of the dataset
    df = df.replace({"?": np.nan, "None": np.nan, "": np.nan})
    return df


df = load_raw(DATA_PATH)
print("Shape (raw):", df.shape)
df.head()


In [ ]:
# Target: early readmission within 30 days (binary, per coursework framing)
# Positive = readmitted in < 30 days; negative = NO or >30 days
df["readmitted_binary"] = (df["readmitted"] == "<30").astype(int)
print(df["readmitted"].value_counts())
print("\nBinary (<30 vs not):", df["readmitted_binary"].value_counts(normalize=True).round(3))


In [ ]:
# Inspect missingness (before cleaning)
miss = df.isna().mean().sort_values(ascending=False)
print("Top missing % (columns):")
print((miss * 100).head(15).round(2))


In [ ]:
def clean_diabetes_df(frame: pd.DataFrame) -> pd.DataFrame:
    d = frame.copy()
    # Identifiers: keep patient_nbr for grouping; drop encounter_id from features later
    # Weight: overwhelmingly missing in this CSV — drop (justify in report)
    if "weight" in d.columns:
        d = d.drop(columns=["weight"])
    # Zero-variance medication flags from data dictionary
    for c in ["examide", "citoglipton"]:
        if c in d.columns:
            d = d.drop(columns=[c])
    # Very sparse admin text fields — drop to control dimensionality + noise
    for c in ["payer_code", "medical_specialty"]:
        if c in d.columns:
            d = d.drop(columns=[c])
    # Simplify diagnosis codes (high cardinality)
    for c in ["diag_1", "diag_2", "diag_3"]:
        if c in d.columns:
            d[c] = d[c].fillna("unknown").astype(str).str.strip().str[:4]
    # Fill remaining object columns with explicit missing bucket
    for c in d.select_dtypes(include=["object"]).columns:
        if c in ("readmitted",):
            continue
        d[c] = d[c].fillna("Missing")
    return d


df_clean = clean_diabetes_df(df)
print("Shape after cleaning drops/fills:", df_clean.shape)


In [ ]:
# Remove near-constant columns (no predictive information)
row_n = len(df_clean)
const_cols = []
for c in df_clean.columns:
    if c in ("readmitted", "readmitted_binary", "patient_nbr", "encounter_id"):
        continue
    nu = df_clean[c].nunique(dropna=False)
    if nu <= 1:
        const_cols.append(c)
    elif df_clean[c].value_counts(normalize=True, dropna=False).iloc[0] > 0.999:
        const_cols.append(c)

if const_cols:
    df_clean = df_clean.drop(columns=const_cols)
    print("Dropped near-constant:", const_cols)
else:
    print("No near-constant columns found.")


In [ ]:
# Feature / target matrices (patient-level split to reduce leakage)
ID_COL = "patient_nbr"
y = df_clean["readmitted_binary"].astype(int)
groups = df_clean[ID_COL]

feature_drop = ["encounter_id", ID_COL, "readmitted", "readmitted_binary"]
X = df_clean.drop(columns=[c for c in feature_drop if c in df_clean.columns])

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
X.head()


In [ ]:
# Train / test split — GroupShuffleSplit by patient
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

print("Train rows:", len(X_train), "Test rows:", len(X_test))
print("Train positive rate:", y_train.mean().round(4))
print("Test positive rate:", y_test.mean().round(4))


In [ ]:
# Outliers on key numeric utilisation features (winsorise at 99th percentile, train only)
def winsorise_train_test(
    X_tr: pd.DataFrame, X_te: pd.DataFrame, cols: list[str], q: float = 0.99
) -> tuple[pd.DataFrame, pd.DataFrame]:
    tr, te = X_tr.copy(), X_te.copy()
    for c in cols:
        if c not in tr.columns:
            continue
        cap = tr[c].quantile(q)
        tr[c] = tr[c].clip(upper=cap)
        te[c] = te[c].clip(upper=cap)
    return tr, te


util_cols = [
    c
    for c in [
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
    ]
    if c in X_train.columns
]
X_train, X_test = winsorise_train_test(X_train, X_test, util_cols)
print("Winsorised columns:", util_cols)


### Part 1 — Required visualisations

In [ ]:
# 1) Target distribution — donut (3-class) + horizontal count bars (binary)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
c3 = df_clean["readmitted"].value_counts()
pal3 = sns.color_palette("Spectral", n_colors=len(c3))
axes[0].pie(
    c3.values,
    labels=c3.index,
    autopct="%1.1f%%",
    colors=pal3,
    pctdistance=0.78,
    wedgeprops=dict(width=0.45, edgecolor="white"),
    startangle=90,
)
axes[0].set_title("Readmission label mix (donut)")
cb = df_clean["readmitted_binary"].value_counts().sort_index()
lbl = ["Not early (<30d)", "Early readmit (<30d)"]
axes[1].barh(lbl, cb.values, color=["#3a86ff", "#ff006e"], height=0.45, edgecolor="black", linewidth=0.6)
axes[1].invert_yaxis()
axes[1].set_xlabel("Encounters")
for i, v in enumerate(cb.values):
    axes[1].text(v + max(cb.values) * 0.02, i, f"{v:,}", va="center", fontsize=10)
axes[1].set_title("Binary target (horizontal bar)")
plt.tight_layout()
plt.show()


In [ ]:
# 2) Age vs readmission — point estimates with 95% CI (not a plain bar chart)
age_order = sorted(df_clean["age"].unique(), key=lambda s: int(s.strip("[]").split("-")[0]))
plt.figure(figsize=(10, 4.5))
sns.pointplot(
    data=df_clean,
    x="age",
    y="readmitted_binary",
    order=age_order,
    errorbar=("ci", 95),
    capsize=0.06,
    color="#1b998b",
    markers="D",
    linestyles="-",
    linewidth=1.8,
)
plt.ylabel("Mean P(early readmit)")
plt.xlabel("Age band")
plt.title("Age trend with bootstrap-style CI (Seaborn pointplot)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# 3) Medications vs readmission — letter-value (boxen) plot
plt.figure(figsize=(9, 4.5))
sns.boxenplot(
    data=df_clean,
    x="readmitted_binary",
    y="num_medications",
    hue="readmitted_binary",
    palette=["#5c4d7d", "#d1495b"],
    dodge=False,
    legend=False,
    k_depth="proportion",
)
plt.xlabel("Binary target (0 = not <30d, 1 = <30d)")
plt.ylabel("Number of medications")
plt.title("Distribution shape: medications by readmission (boxenplot)")
plt.tight_layout()
plt.show()


In [ ]:
# 4) Correlation structure — clustered heatmap on a numeric subset (reorders rows/cols)
sub_num = numeric_features[: min(14, len(numeric_features))]
corr_sub = X_train[sub_num].corr()
if len(sub_num) >= 2:
    grid = sns.clustermap(
        corr_sub,
        cmap="vlag",
        center=0,
        figsize=(9.5, 9.5),
        dendrogram_ratio=0.1,
        cbar_kws={"label": "Pearson r"},
        linewidths=0.4,
        linecolor="white",
    )
    grid.fig.suptitle("Hierarchically clustered correlations (train, numeric subset)", y=1.02, fontsize=12)
    plt.show()
else:
    plt.figure(figsize=(6, 4))
    sns.heatmap(corr_sub, cmap="vlag", center=0, annot=True)
    plt.title("Correlation (small numeric set)")
    plt.tight_layout()
    plt.show()
corr = X_train[numeric_features].corr()
stack = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
strong = stack.stack()
strong = strong[strong > 0.5]
print("Strong correlations (|r|>0.5, full numeric set):\n", strong.head(20))


In [ ]:
# 5) Extra EDA — violin: length of stay by binary outcome (training patients)
plt.figure(figsize=(8, 4.5))
sns.violinplot(
    data=df_clean.iloc[train_idx],
    x="readmitted_binary",
    y="time_in_hospital",
    palette=["#457b9d", "#e63946"],
    inner="quart",
    cut=0,
)
plt.xlabel("Binary target")
plt.ylabel("Time in hospital (days)")
plt.title("LOS distribution by early readmission (violin + quartiles)")
plt.tight_layout()
plt.show()


In [ ]:
# Persist cleaned table for PySpark / report reproducibility
CLEAN_PATH = ROOT / "data_clean_group12.parquet"
try:
    df_clean.to_parquet(CLEAN_PATH, index=False)
    print("Saved:", CLEAN_PATH)
except Exception as e:
    print("Parquet save skipped (install pyarrow if needed):", e)


## Part 2 — Initial model building (scikit-learn)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from imblearn.under_sampling import RandomUnderSampler

try:
    import google.colab  # noqa: F401

    _COLAB_FAST = True
except ImportError:
    _COLAB_FAST = False
# Colab: fewer CV folds + smaller forest = much faster; local keeps full settings
CV_SPLITS = 3 if _COLAB_FAST else 5
RF_TREES = 80 if _COLAB_FAST else 200
LR_ITER = 800 if _COLAB_FAST else 2000
if _COLAB_FAST:
    print("Colab fast mode: CV_SPLITS=", CV_SPLITS, "RF_TREES=", RF_TREES)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", sparse_output=True, max_categories=25),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Fit preprocessor on training data only
preprocessor.fit(X_train)
X_tr = preprocessor.transform(X_train)
X_te = preprocessor.transform(X_test)
print("Train matrix:", X_tr.shape, "Test matrix:", X_te.shape)


In [ ]:
# Random undersampling: all minority + equal random majority (training only)
rus = RandomUnderSampler(random_state=RANDOM_STATE, sampling_strategy="auto")
X_tr_bal, y_tr_bal = rus.fit_resample(X_tr, y_train)
print("After undersampling — shape:", X_tr_bal.shape, "positive rate:", y_tr_bal.mean().round(3))


In [ ]:
def evaluate_classifier(name, model, Xb, yb, X_eval, y_eval, cv_splits=CV_SPLITS):
    skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    cv_auc = cross_val_score(model, Xb, yb, cv=skf, scoring="roc_auc", n_jobs=-1)
    model.fit(Xb, yb)
    prob = model.predict_proba(X_eval)[:, 1]
    pred = (prob >= 0.5).astype(int)
    return {
        "model": name,
        "cv_roc_auc_mean": float(np.mean(cv_auc)),
        "cv_roc_auc_std": float(np.std(cv_auc)),
        "test_accuracy": accuracy_score(y_eval, pred),
        "test_precision": precision_score(y_eval, pred, zero_division=0),
        "test_recall": recall_score(y_eval, pred, zero_division=0),
        "test_f1": f1_score(y_eval, pred, zero_division=0),
        "test_roc_auc": roc_auc_score(y_eval, prob),
        "fitted": model,
        "prob": prob,
    }


models = {
    "LogisticRegression": LogisticRegression(
        max_iter=LR_ITER,
        class_weight=None,
        random_state=RANDOM_STATE,
        solver="saga",
        n_jobs=-1,
    ),
    "DecisionTreeClassifier": DecisionTreeClassifier(
        max_depth=12,
        min_samples_leaf=40,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    ),
    "RandomForestClassifier": RandomForestClassifier(
        n_estimators=RF_TREES,
        max_depth=18,
        min_samples_leaf=20,
        random_state=RANDOM_STATE,
        class_weight="balanced_subsample",
        n_jobs=-1,
    ),
}

results = []
for nm, clf in models.items():
    results.append(evaluate_classifier(nm, clf, X_tr_bal, y_tr_bal, X_te, y_test))

metrics_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ("fitted", "prob")} for r in results])
metrics_df


In [ ]:
# ROC + Precision–Recall (test set) — fast: curves from stratified subsample only
# (metrics_df / test_roc_auc already used the full test set in the cell above)
from sklearn.model_selection import train_test_split

MAX_CURVE_POINTS = 6_000
y_te_arr = np.asarray(y_test)
idx = np.arange(len(y_te_arr))
if len(idx) > MAX_CURVE_POINTS:
    _, plot_ix = train_test_split(
        idx, train_size=MAX_CURVE_POINTS, stratify=y_te_arr, random_state=RANDOM_STATE
    )
else:
    plot_ix = idx
y_curve = y_te_arr[plot_ix]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for r in results:
    p = np.asarray(r["prob"])[plot_ix]
    RocCurveDisplay.from_predictions(y_curve, p, name=r["model"], ax=axes[0])
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.45, label="chance")
axes[0].set_title("ROC (test — plot subsample for speed)")
axes[0].legend(loc="lower right", fontsize=8)
for r in results:
    p = np.asarray(r["prob"])[plot_ix]
    PrecisionRecallDisplay.from_predictions(y_curve, p, name=r["model"], ax=axes[1])
axes[1].set_title("Precision–Recall (test — plot subsample)")
plt.tight_layout()
plt.show()

melted = metrics_df.melt(
    id_vars=["model"],
    value_vars=["test_precision", "test_recall", "test_f1"],
    var_name="metric",
    value_name="value",
)
melted["metric"] = melted["metric"].str.replace("test_", "")
plt.figure(figsize=(9, 4))
sns.barplot(data=melted, x="model", y="value", hue="metric", palette="Dark2", errorbar=None)
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.title("Test precision / recall / F1 compared across classifiers")
plt.legend(title="")
plt.tight_layout()
plt.show()


In [ ]:
best = max(results, key=lambda r: r["test_roc_auc"])
print("Best test ROC-AUC:", best["model"], round(best["test_roc_auc"], 4))
global_sklearn_model = best["fitted"]


## Part 3 — Distributed analysis (PySpark) + clustering

In [ ]:
# Dense feature matrix for Part 3 — float32 + optional row cap (Colab RAM)
import os


def _part3_ram_cap_df(d: pd.DataFrame) -> pd.DataFrame:
    raw = os.environ.get("PART3_MAX_ROWS", "").strip()
    if raw == "":
        cap = -1
    else:
        cap = int(raw)
    if cap < 0:
        try:
            import google.colab  # noqa: F401

            cap = 18_000
        except ImportError:
            cap = 0
    if cap == 0 or len(d) <= cap:
        return d
    from sklearn.model_selection import train_test_split

    ix, _ = train_test_split(
        np.arange(len(d)),
        train_size=cap,
        stratify=d["readmitted_binary"].astype(int),
        random_state=RANDOM_STATE,
    )
    print("Part 3: Colab/RAM subsample —", cap, "stratified rows (of", len(d), ")")
    return d.iloc[ix].reset_index(drop=True)


df_p3 = _part3_ram_cap_df(df_clean.copy())
pdf_model = df_p3.drop(columns=[c for c in ["readmitted"] if c in df_p3.columns]).copy()
pdf_model["label"] = pdf_model["readmitted_binary"].astype(float)
drop_spark = [c for c in ["encounter_id", "patient_nbr", "readmitted_binary"] if c in pdf_model.columns]
X_cols = [c for c in pdf_model.columns if c not in drop_spark + ["label"]]
X_pd = pd.get_dummies(pdf_model[X_cols], drop_first=True).fillna(0).astype("float32")
print("Part 3 design matrix:", X_pd.shape, "(float32)")
# Keep raw dense matrix + labels in the same cell as X_pd so later cells never hit NameError
# if this cell was skipped or stopped before get_dummies finished.
local_X = X_pd.values
y_readmit = pdf_model["readmitted_binary"].astype(int).values


In [ ]:
# Elbow + silhouette to choose k (sample for speed; document in report)
from sklearn.cluster import KMeans as SKKMeans
from sklearn.metrics import silhouette_score

try:
    import google.colab  # noqa: F401

    sample_n = min(8000, X_pd.shape[0])
except ImportError:
    sample_n = min(15000, X_pd.shape[0])
idx = rng.choice(X_pd.shape[0], size=sample_n, replace=False)
X_samp = X_pd.iloc[idx].values

Ks = range(2, 11)
inertias, sil_scores = [], []
for k in Ks:
    km = SKKMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    lab = km.fit_predict(X_samp)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_samp, lab))

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.5))
k_list = list(Ks)
ax[0].fill_between(k_list, inertias, alpha=0.35, color="steelblue")
ax[0].plot(k_list, inertias, "o-", color="navy", lw=2, markersize=7)
ax[0].set_xlabel("k")
ax[0].set_ylabel("Inertia (within-cluster SSE)")
ax[0].set_title("Elbow curve with shaded inertia")

if len(inertias) > 1:
    deltas = -np.diff(inertias)
    ax[1].bar(range(len(deltas)), deltas, tick_label=[f"{k_list[i]}→{k_list[i+1]}" for i in range(len(deltas))], color="teal", edgecolor="black", alpha=0.85)
    ax[1].set_xlabel("Step in k")
    ax[1].set_ylabel("Inertia drop (marginal gain)")
    ax[1].set_title("Marginal SSE reduction when adding a cluster")
    ax[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 4))
colors = plt.cm.magma(np.linspace(0.25, 0.92, len(k_list)))
plt.barh([str(k) for k in k_list], sil_scores, color=colors, edgecolor="black", linewidth=0.5)
plt.axvline(x=max(sil_scores), color="limegreen", ls="--", lw=1.5, label="best silhouette")
plt.xlabel("Mean silhouette coefficient")
plt.ylabel("k")
plt.title("Silhouette by k (horizontal bars)")
plt.legend()
plt.tight_layout()
plt.show()

K_CLUSTERS = list(Ks)[int(np.argmax(sil_scores))]
print("Chosen k (max silhouette on sample):", K_CLUSTERS)


In [ ]:
# PySpark path when Java is available; otherwise sklearn mirrors the same workflow
import os
from pathlib import Path

import numpy as np
import pandas as pd

if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 42
if "rng" not in globals():
    rng = np.random.RandomState(RANDOM_STATE)


def _part3_ram_cap_df(d: pd.DataFrame) -> pd.DataFrame:
    raw = os.environ.get("PART3_MAX_ROWS", "").strip()
    if raw == "":
        cap = -1
    else:
        cap = int(raw)
    if cap < 0:
        try:
            import google.colab  # noqa: F401

            cap = 18_000
        except ImportError:
            cap = 0
    if cap == 0 or len(d) <= cap:
        return d
    from sklearn.model_selection import train_test_split

    ix, _ = train_test_split(
        np.arange(len(d)),
        train_size=cap,
        stratify=d["readmitted_binary"].astype(int),
        random_state=RANDOM_STATE,
    )
    print("Part 3: Colab/RAM subsample —", cap, "stratified rows (of", len(d), ")")
    return d.iloc[ix].reset_index(drop=True)


# Rebuild Part 3 dense inputs if missing (Colab restart, "Run this cell" only, etc.)
def _load_csv_part3(path: Path) -> pd.DataFrame:
    d = pd.read_csv(path, low_memory=False)
    return d.replace({"?": np.nan, "None": np.nan, "": np.nan})


def _clean_part3(d: pd.DataFrame) -> pd.DataFrame:
    d = d.copy()
    if "weight" in d.columns:
        d = d.drop(columns=["weight"])
    for c in ["examide", "citoglipton"]:
        if c in d.columns:
            d = d.drop(columns=[c])
    for c in ["payer_code", "medical_specialty"]:
        if c in d.columns:
            d = d.drop(columns=[c])
    for c in ["diag_1", "diag_2", "diag_3"]:
        if c in d.columns:
            d[c] = d[c].fillna("unknown").astype(str).str.strip().str[:4]
    for c in d.select_dtypes(include=["object"]).columns:
        if c in ("readmitted",):
            continue
        d[c] = d[c].fillna("Missing")
    return d


def _drop_near_constant(d: pd.DataFrame) -> pd.DataFrame:
    const_cols = []
    for c in d.columns:
        if c in ("readmitted", "readmitted_binary", "patient_nbr", "encounter_id"):
            continue
        nu = d[c].nunique(dropna=False)
        if nu <= 1:
            const_cols.append(c)
        elif d[c].value_counts(normalize=True, dropna=False).iloc[0] > 0.999:
            const_cols.append(c)
    return d.drop(columns=const_cols) if const_cols else d


if "X_pd" not in globals() or "local_X" not in globals() or "y_readmit" not in globals():
    if "df_clean" not in globals():
        base = None
        if "df" in globals():
            base = globals()["df"]
            if not isinstance(base, pd.DataFrame):
                base = None
        if base is not None:
            if "readmitted_binary" not in base.columns:
                base = base.copy()
                base["readmitted_binary"] = (base["readmitted"] == "<30").astype(int)
            df_clean = _drop_near_constant(_clean_part3(base))
            print("Built df_clean from in-memory df — shape:", df_clean.shape)
        else:
            candidates = []
            if "DATA_PATH" in globals():
                candidates.append(Path(str(globals()["DATA_PATH"])))
            candidates.extend(
                [Path.cwd() / "diabetic_data.csv", Path("/content/diabetic_data.csv"), Path("diabetic_data.csv")]
            )
            loaded = None
            for p in candidates:
                if p.exists():
                    loaded = _load_csv_part3(p)
                    print("Loaded CSV for Part 3:", p.resolve())
                    break
            if loaded is None:
                raise RuntimeError(
                    "No df_clean or df, and diabetic_data.csv not found. "
                    "Run Colab setup + Part 1, or put diabetic_data.csv in /content or the notebook folder."
                )
            loaded["readmitted_binary"] = (loaded["readmitted"] == "<30").astype(int)
            df_clean = _drop_near_constant(_clean_part3(loaded))
            print("Built df_clean from file — shape:", df_clean.shape)

    df_p3 = _part3_ram_cap_df(df_clean.copy())
    pdf_model = df_p3.drop(columns=[c for c in ["readmitted"] if c in df_p3.columns]).copy()
    pdf_model["label"] = pdf_model["readmitted_binary"].astype(float)
    drop_spark = [c for c in ["encounter_id", "patient_nbr", "readmitted_binary"] if c in pdf_model.columns]
    X_cols = [c for c in pdf_model.columns if c not in drop_spark + ["label"]]
    X_pd = pd.get_dummies(pdf_model[X_cols], drop_first=True).fillna(0).astype("float32")
    print("Part 3 design matrix:", X_pd.shape, "(float32)")
    local_X = X_pd.values
    y_readmit = pdf_model["readmitted_binary"].astype(int).values

if "K_CLUSTERS" not in globals():
    K_CLUSTERS = 3
    print("K_CLUSTERS was not set — using default K_CLUSTERS = 3 (run elbow cell for a data-driven k).")

os.environ.setdefault(
    "PYSPARK_SUBMIT_ARGS",
    "--driver-java-options=-Dlog4j.logger.org.apache.spark=WARN pyspark-shell",
)

PART3_BACKEND = None
spark = None

try:
    from pyspark.sql import SparkSession
    from pyspark.ml.feature import VectorAssembler, StandardScaler as SparkStandardScaler
    from pyspark.ml.classification import LogisticRegression as SparkLR
    from pyspark.ml.clustering import KMeans
    from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

    spark = (
        SparkSession.builder.appName("CO7093_Group12")
        .master("local[*]")
        .config("spark.sql.shuffle.partitions", "8")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("WARN")

    spark_df = spark.createDataFrame(
        pd.concat([X_pd, pdf_model[["label"]].reset_index(drop=True)], axis=1)
    )
    feature_cols_spark = [c for c in spark_df.columns if c != "label"]
    assembler = VectorAssembler(inputCols=feature_cols_spark, outputCol="features_raw")
    scaler_sp = SparkStandardScaler(
        inputCol="features_raw", outputCol="features", withStd=True, withMean=True
    )
    assembled = assembler.transform(spark_df)
    scaler_model = scaler_sp.fit(assembled)
    spark_data = scaler_model.transform(assembled).select("features", "label")
    spark_data.cache()
    print("PySpark rows:", spark_data.count(), "| features:", len(feature_cols_spark))

    train_sp, test_sp = spark_data.randomSplit([0.8, 0.2], seed=RANDOM_STATE)
    lr_spark = SparkLR(
        featuresCol="features", labelCol="label", maxIter=80, regParam=0.01, elasticNetParam=0.0
    )
    model_sp = lr_spark.fit(train_sp)
    pred_sp = model_sp.transform(test_sp)
    bin_ev = BinaryClassificationEvaluator(
        rawPredictionCol="rawPrediction", labelCol="label", metricName="areaUnderROC"
    )
    acc_ev = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="accuracy"
    )
    print(
        "PySpark LogisticRegression — ROC-AUC:",
        round(bin_ev.evaluate(pred_sp), 4),
        " accuracy:",
        round(acc_ev.evaluate(pred_sp), 4),
    )

    kmeans = KMeans(
        featuresCol="features",
        predictionCol="cluster",
        k=K_CLUSTERS,
        seed=RANDOM_STATE,
        maxIter=40,
    )
    kmodel = kmeans.fit(spark_data)
    clustered = kmodel.transform(spark_data)
    clustered.groupBy("cluster").count().orderBy("cluster").show()

    local_df = clustered.select("cluster", "label").toPandas()
    local_df["readmitted_binary"] = y_readmit
    PART3_BACKEND = "pyspark"
    print("Part 3 executed with PySpark (distributed).")
    from sklearn.preprocessing import StandardScaler as SklearnScaler

    local_X_m = SklearnScaler().fit_transform(np.asarray(local_X, dtype=np.float32)).astype(np.float32)
    import gc

    try:
        del X_pd
        del local_X
    except NameError:
        pass
    gc.collect()

except Exception as exc:
    print("PySpark unavailable on this machine (Java/JVM required):", exc)
    print(">>> Using scikit-learn fallback: same steps locally for marking/reproducibility.")
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression as SKLR
    from sklearn.cluster import KMeans as SKKMeans2
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score
    import gc

    _glr_iter = 2000
    try:
        import google.colab  # noqa: F401

        _glr_iter = 600
    except ImportError:
        pass

    lx = np.asarray(local_X, dtype=np.float32, order="C")
    Xs = StandardScaler().fit_transform(lx)
    del lx
    X_tr, X_te, y_tr, y_te = train_test_split(
        Xs, y_readmit, test_size=0.2, random_state=RANDOM_STATE, stratify=y_readmit
    )
    glr = SKLR(max_iter=_glr_iter, class_weight="balanced", random_state=RANDOM_STATE, solver="saga")
    glr.fit(X_tr, y_tr)
    prob_te = glr.predict_proba(X_te)[:, 1]
    print(
        "sklearn LogisticRegression (dense matrix) — ROC-AUC:",
        round(roc_auc_score(y_te, prob_te), 4),
    )

    sub_n = min(8000, Xs.shape[0])
    sub_ix = rng.choice(Xs.shape[0], size=sub_n, replace=False)
    km_full = SKKMeans2(
        n_clusters=K_CLUSTERS, random_state=RANDOM_STATE, n_init=3, max_iter=40
    )
    km_full.fit(Xs[sub_ix])
    cl_lab = km_full.predict(Xs)
    local_df = pd.DataFrame({"cluster": cl_lab, "readmitted_binary": y_readmit})
    _u, cnts = np.unique(cl_lab, return_counts=True)
    print("Cluster counts (sklearn K-Means):")
    print(pd.Series(cnts, index=_u))
    PART3_BACKEND = "sklearn_fallback"
    local_X_m = np.asarray(Xs, dtype=np.float32)
    del Xs
    try:
        del X_pd
        del local_X
    except NameError:
        pass
    gc.collect()


In [ ]:
# Local classifiers per cluster vs global LR (standardised dense features)
from sklearn.linear_model import LogisticRegression as LRLocal
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

if "local_X_m" not in globals():
    local_X_m = StandardScaler().fit_transform(np.asarray(local_X, dtype=np.float32)).astype(np.float32)
else:
    print("Reusing local_X_m from Part 3 cell (avoids a second full-matrix scale + saves RAM).")

cluster_models, cluster_metrics = {}, []

for cl in sorted(local_df["cluster"].unique()):
    mask = local_df["cluster"].values == cl
    if mask.sum() < 500:
        continue
    X_c = local_X_m[mask]
    y_c = local_df["readmitted_binary"].values[mask]
    n = len(y_c)
    cut = int(0.8 * n)
    idx_perm = rng.permutation(n)
    tr, te = idx_perm[:cut], idx_perm[cut:]
    if y_c[tr].sum() < 5 or y_c[tr].sum() == len(y_c[tr]):
        continue
    lr_c = LRLocal(max_iter=1500, class_weight="balanced", random_state=RANDOM_STATE)
    lr_c.fit(X_c[tr], y_c[tr])
    prob_c = lr_c.predict_proba(X_c[te])[:, 1]
    try:
        auc_c = roc_auc_score(y_c[te], prob_c)
    except ValueError:
        auc_c = float("nan")
    cluster_models[int(cl)] = lr_c
    cluster_metrics.append({"cluster": int(cl), "n": int(mask.sum()), "holdout_roc_auc": auc_c})

cm_df = pd.DataFrame(cluster_metrics)
print("Per-cluster logistic models (80/20 inside cluster):")
print(cm_df)

global_lr_dense = LRLocal(max_iter=1500, class_weight="balanced", random_state=RANDOM_STATE)
g_cut = int(0.8 * len(local_df))
gp = rng.permutation(len(local_df))
gtr, gte = gp[:g_cut], gp[g_cut:]
global_lr_dense.fit(local_X_m[gtr], local_df["readmitted_binary"].values[gtr])
prob_g = global_lr_dense.predict_proba(local_X_m[gte])[:, 1]
auc_global_dense = roc_auc_score(local_df["readmitted_binary"].values[gte], prob_g)
print("Global LR holdout ROC-AUC (same features):", round(auc_global_dense, 4))

if len(cm_df) > 0:
    plt.figure(figsize=(8, 4))
    sns.barplot(data=cm_df, x="cluster", y="holdout_roc_auc", hue="cluster", palette="husl", legend=False)
    plt.axhline(auc_global_dense, color="crimson", ls="--", lw=2, label="Global LR AUC")
    plt.ylabel("Holdout ROC-AUC")
    plt.xlabel("Cluster id")
    plt.title("Local cluster models vs global baseline")
    plt.legend()
    plt.tight_layout()
    plt.show()

print("Backend used for distributed section:", PART3_BACKEND)


In [ ]:
# PCA 2D with hexbin density + cluster colours (sample)
from sklearn.decomposition import PCA

sample_v = min(8000, local_X_m.shape[0])
ix = rng.choice(local_X_m.shape[0], size=sample_v, replace=False)
pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
Z = pca2.fit_transform(local_X_m[ix])
lab_c = local_df["cluster"].values[ix]

fig, ax = plt.subplots(figsize=(8, 6.5))
ax.hexbin(Z[:, 0], Z[:, 1], gridsize=38, cmap="bone_r", mincnt=1, alpha=0.42)
sc = ax.scatter(
    Z[:, 0],
    Z[:, 1],
    c=lab_c,
    cmap="tab10",
    s=28,
    alpha=0.9,
    edgecolors="white",
    linewidths=0.25,
)
plt.colorbar(sc, ax=ax, label="cluster id", fraction=0.046, pad=0.02)
ax.set_title("PCA: hexbin density + K-Means cluster overlay (" + str(PART3_BACKEND) + ")")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()


In [ ]:
if spark is not None:
    spark.stop()
    print("Spark session stopped.")
else:
    print("No Spark session (sklearn fallback was used).")


### How the completed pipeline behaves
1. **Load** `diabetic_data.csv` and map `?` / `None` to missing.
2. **Define** binary target: `readmitted == '<30'` → positive class.
3. **Clean** aggressively: drop `weight`, sparse admin text fields, zero-variance drugs, simplify diagnosis strings, impute categorical gaps.
4. **Split** with **GroupShuffleSplit** on `patient_nbr` so the same patient does not appear in both train and test.
5. **Winsorise** heavy-tailed utilisation counts on the training quantiles.
6. **EDA plots:** donut + horizontal bars (target), **pointplot** with CI (age), **boxenplot** (medications), **clustermap** (correlation subset), **violinplot** (LOS). **Part 2** adds **PR curves**, grouped **metric bar chart**, then **ROC**.
7. **Part 2:** `ColumnTransformer` → median imputation + scaling (numeric) and one-hot (categorical, capped categories). **RandomUnderSampler** balances the **training** design matrix. Three classifiers with **5-fold stratified CV ROC-AUC** on the balanced train, then **test** metrics including **ROC-AUC** (realistic imbalance in test).
8. **Part 3:** Dense matrix via `pd.get_dummies` on **all rows**. With **Java/PySpark** available: **Spark** `StandardScaler` + **LogisticRegression** + **KMeans** on the full table. If Spark cannot start (no JVM), the notebook **falls back** to the same logic with **scikit-learn** so it still runs for development; use PySpark on the lab machine for the assessed “distributed” requirement.
9. **Local vs global:** Per-cluster and global logistic models use **standardised** dense features; compare ROC-AUC on holdout splits and discuss heterogeneity.

**Note:** Per-cluster metrics use internal holdout splits; report should relate these to cluster sizes and purity, and discuss when local models help (heterogeneous risk patterns).
